# Step 1.1.d — BIO variables of ERA5 (1981–2015), EC-Earth3 grid, global, MIRCA2000 no threshold

The 12 BIO variables of each crop, computed on ERA5 regridded onto the EC-Earth3 grid (CDO), over the growing
season given by MIRCA2000 (no-threshold presence and calendar, notebook 0.1), on the **land cells** of the land mask of
0.1 (45 112 cells), globally. Same BIO definitions and harvest years (1982–2015) as before; what changes:

- grid: EC-Earth3 (0.7°, global) instead of the 1° Europe grid; ERA5 is read through the configuration
  (`crop_utils.read_climate`: units, longitudes in −180…180, grid checked) and only its land cells are kept, so the
  daily arrays are (days × land cells): about 2.3 GB per variable;
- the climate (reading, ET0) is computed once, then the BIO variables of every crop of `CROPS` in a loop;
- calendar gap-filling (land cells without the crop): inverse-distance weighting of the 5 nearest cropped cells as
  before, with great-circle distances (the former lon × cos 55° scaling was for Europe only).

**Phases: three variants, to be compared by the SVM** (MIRCA2000 gives only the sowing and harvest months, ±15 days,
and not the variety, so the split of the season is an assumption; the test tells how much the results depend on it):

| variant | phase 1 / phase 2 | variables |
|---|---|---|
| `A_60d` | phase 2 = the last 60 days of the season, for every crop (Ceglar et al. 2021, durum wheat) | BIO01–BIO12 |
| `B_season` | no split: the whole season, sowing → harvest | BIO01–BIO05 (season means), BIO11, BIO12 |
| `C_crop` | phase 2 from the literature per crop: wheat the last 40 days (anthesis → maturity, grain filling ≈ 39 days, Embrapa), maize the last 60 days (silking R1 → maturity R6, 55–65 days, Abendroth et al. 2011), potato from 50 days after planting (tuber initiation → harvest: bulking and maturation, CIP, UF/IFAS), soybean the last 70 days (beginning bloom R1 → maturity R7, ≈ 64–80 days, Clemson and Purdue extension), rice the last 30 days (flowering → maturity, tropics) | BIO01–BIO12 |

ET0: Hargreaves. Radiation: W m-2 × 0.0864 → MJ m-2 day-1.

**Radiation test**: the global (surface) solar radiation is used only by BIO03 and BIO08 (ET0 uses the extraterrestrial
radiation Ra), and it is costly to obtain for the GCM runs. Next to BIO03 / BIO08 from ERA5 (ssrd), each table also
has **BIO03e / BIO08e**: the same means of the radiation **estimated from temperature** with FAO-56 (Allen et al. 1998,
eq. 50; Hargreaves & Samani 1982), Rs = kRs √(Tmax − Tmin) Ra, kRs = 0.16 inland and 0.19 in coastal cells (land
cells next to the sea), capped at the clear-sky radiation 0.75 Ra (eq. 37, elevation neglected). The SVM (3.g)
compares real, estimated and no radiation.

**Two steps**: (1) anthesis test, `VARIANTS_RUN = VARIANTS`, `RAD_EST = False`; (2) once 3.g has chosen the variant,
radiation test, `VARIANTS_RUN = ["<chosen variant>"]`, `RAD_EST = True` (recomputes that variant with BIO03e / BIO08e).

**SVM training domain** (`TRAIN_DOMAIN = "mirca_cropland"`): the training table keeps only the **MIRCA2000 cropland**
cells (harvested area of any of the 26 classes, notebook 0.1): presences (label 1) and absences where other crops
are grown (label 0), a target-group background (Phillips et al. 2009), so that an absence reflects the crop and not
the land use (forest, pasture, ice, desert). This is the cropland of Ramankutty et al. (2008), the 'arable land' of
Ceglar et al. (2021); checked against ESA CCI Land Cover 2000 in 0.1.b (99.9 % of MIRCA's area in ESA cropland cells).
The BIO variables are still computed on **all land cells**: the SVM is applied to all of them (prediction).

**Presence rule (`CELL_AREA_THR`, route cellareafrac, as notebook 0.3 in Europe)**: 5' pixel without threshold (any
harvested area counts), EC-Earth3 cell = **true area fraction**: harvested area of the crop / land area of the cell
above `CELL_AREA_THR` (0.05 %). The **same rule for the background**: a cell is cropland if the area of all the crops
is above 0.05 % of its land area. A trace of cultivation in a 0.7° cell is not representative of the cell climate
(e.g. valleys in the mountains of eastern Tibet, 0.002 % of the cell in maize, cell mean temperature of the season
−10 °C): negligible for the crop, it is negligible for the cropland too. Cells of the background with only a trace of
the crop become absences; the MIRCA calendar of every cell with the crop is kept (gap-filling only elsewhere).
`CELL_AREA_THR = None`: route no_threshold (any area), used for tests 1–3 of 3.g; the threshold is test 4 (1.1.d
rerun with `CELL_AREA_THR = 0.0005` for the chosen variant only). Tested on EC-Earth3 1981–82: background 88 % of the cells and
99.99 % of the cropland area, 99.7 % of the maize area kept as presence; maize "suitable" in Antarctica 10 % → 0 %.

Outputs in `<out_root>/1_biovariables/ERA5_<route>/<variant>/` (route `mirca2000_no_threshold` or `mirca2000_cellareafrac0.05pct`) (BDAP: `elisa/Crop_suitability/outputs/...`),
per crop: `BIOvariables_ERA5_1981_2015_ecearth3_<crop>.nc` (BIO variables on all land cells, the calendar used,
presence and the training domain), `training_dataset_ERA5_ecearth3_<crop>.csv` (one row per cell of the training
domain: lat, lon, BIO variables, label 1 = MIRCA2000 presence) and a map of the variables.

In [ ]:
import os; os.environ.setdefault("HDF5_USE_FILE_LOCKING", "FALSE")   # BDAP/EOS refuses HDF5 file locks ("NetCDF: HDF error")
import time, datetime
import numpy as np, pandas as pd, xarray as xr
import matplotlib.pyplot as plt
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')

BASE = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "0_Input_data").is_dir())  # racine du projet (local ou BDAP)
import sys; sys.path.insert(0, str(BASE)); import crop_utils as cu   # helpers + config reader (project root)
CFG = cu.load_config()   # config/config_<profile>.json, written by 00_config_and_inventory.ipynb

# ---- parameters
CLIMATE = "era5"                  # dataset of the configuration (ERA5 regridded onto the EC-Earth3 grid)
PERIOD = ("1981", "2015")         # daily data read; BIO averaged over the harvest years PERIOD[0]+1 .. PERIOD[1]
CROPS = ["Maize_rf", "Potato_rf", "Soybean_rf", "Rice_rf", "Wheat_rf", "Wheat-winter_rf", "Wheat-spring_rf"]   # rainfed, notebook 0.1
TRAIN_DOMAIN = "mirca_cropland"  # cells of the SVM training table: "mirca_cropland" (MIRCA2000 cropland) | "land" (all)
CELL_AREA_THR = None              # None = route no_threshold (tests 1-3); 0.0005 = route cellareafrac, presence and
                                  # background = area > 0.05 % of the cell land area (test 4); same value in 1.1.d, 3.g, 1.3.a, 4.g
# phase 2 (end of the season) of each variant, per crop (the part of the name before '-' / '_', e.g. Wheat-winter_rf -> Wheat):
# ("before_maturity", d) = the last d days; ("after_sowing", d) = from d days after sowing; None = no split (whole season)
VARIANTS = {
    "A_60d": {"default": ("before_maturity", 60)},        # Ceglar et al. 2021 (durum wheat), all crops
    "B_season": None,                                      # BIO01-05 over sowing -> harvest, BIO11, BIO12
    "C_crop": {"Wheat": ("before_maturity", 40),          # anthesis -> maturity: grain filling ~39 days (Embrapa)
               "Maize": ("before_maturity", 60),          # silking R1 -> maturity R6: 55-65 days (Abendroth et al. 2011)
               "Potato": ("after_sowing", 50),            # tuber initiation -> harvest: bulking + maturation (CIP, UF/IFAS)
               "Soybean": ("before_maturity", 70),        # beginning bloom R1 -> maturity R7: ~64 (Clemson) to ~80 days (Purdue)
               "Rice": ("before_maturity", 30)},          # flowering -> maturity: ~30 days (tropics)
}
VARIANTS_RUN = list(VARIANTS)    # variants computed in this run (step 2 of the tests: only the variant chosen by test 1)
RAD_EST = False                   # True = also BIO03e / BIO08e, radiation estimated with FAO-56 (step 2: radiation test)
ROUTE = cu.route(CELL_AREA_THR)
OUT0 = cu.out_dir(CFG, "1_biovariables", f"{CLIMATE.upper()}_{ROUTE}")
OUT = {v: cu.out_dir(CFG, "1_biovariables", f"{CLIMATE.upper()}_{ROUTE}", v) for v in VARIANTS}
TAG = f"{CLIMATE.upper()}_{PERIOD[0]}_{PERIOD[1]}_ecearth3"

GRID = cu.svm_grid(CFG)
LAND = cu.land_mask(CFG).values
I, J = np.nonzero(LAND)                                  # the land cells
LAT_C, LON_C = GRID["lat"][I], GRID["lon"][J]
NC = I.size
TRAIN = cu.mirca_cropland(CFG, CELL_AREA_THR).values[I, J] if TRAIN_DOMAIN == "mirca_cropland" else np.ones(NC, bool)
print(f"{CLIMATE} {PERIOD[0]}-{PERIOD[1]} | land cells: {NC} of {LAND.size} | training domain ({TRAIN_DOMAIN}, {ROUTE}): "
      f"{int(TRAIN.sum())} cells | variants: {VARIANTS_RUN} | estimated radiation: {RAD_EST} | outputs: {OUT0}")

## 1. Climate at the land cells (read once for all crops)

In [ ]:
t0 = time.time()
CLIM = {}
for v in ("tasmax", "tasmin", "pr", "rad"):
    da = cu.read_climate(CFG, CLIMATE, v, GRID, PERIOD)
    CLIM[v] = cu.land_series(da, (I, J))
    print(f"  {v:6s} {CLIM[v].shape} ({da.attrs['units']}) read in {(time.time() - t0) / 60:.1f} min")
dates = pd.DatetimeIndex(da["time"].values).normalize()
assert all(CLIM[v].shape[0] == dates.size for v in CLIM), "the variables do not have the same days"
TMAX, TMIN, PR = CLIM["tasmax"], CLIM["tasmin"], CLIM["pr"]
SW = CLIM["rad"] * np.float32(0.0864)                    # W m-2 -> MJ m-2 day-1
del CLIM
print(f"{dates[0].date()} -> {dates[-1].date()}, {dates.size} days")

## 2. ET0 — Hargreaves
$$ET_0 = 0.0023 \times R_a \times (T_{mean} + 17.8) \times \sqrt{T_{max} - T_{min}}$$
with $R_a$ the extraterrestrial radiation in mm/day equivalent ($R_a$ in MJ m-2 day-1 divided by the latent heat of
vaporisation), checked against AgERA5 Penman–Monteith in `1.a)_ET0_method.ipynb`.

In [ ]:
def compute_Ra(lat_rad_2d, doy):
    Gsc   = 0.0820
    dr    = 1 + 0.033 * np.cos(2 * np.pi / 365 * doy)
    delta = 0.409 * np.sin(2 * np.pi / 365 * doy - 1.39)
    ws    = np.arccos(np.clip(-np.tan(lat_rad_2d) * np.tan(delta), -1, 1))
    return (24*60/np.pi)*Gsc*dr*(ws*np.sin(lat_rad_2d)*np.sin(delta)
                                  + np.cos(lat_rad_2d)*np.cos(delta)*np.sin(ws))

def compute_ETo_day(tmax_day, tmin_day, lat_rad_2d, doy):
    # BUG FIX: Ra from compute_Ra() is in MJ/m2/day (extraterrestrial radiation); the
    # Hargreaves formula needs it as mm/day equivalent (Ra_mm = Ra / lambda). This division
    # was missing here, inflating ETo by a factor of ~lambda (~2.4-2.5x) -- confirmed against
    # AgERA5 Penman-Monteith reference in 1.a)_ET0_method.ipynb. 2_GAEZ's own Hargreaves
    # notebook already does this correctly; this fix makes 1.a consistent with it.
    Tmean = (tmax_day + tmin_day) / 2.0
    TD    = np.maximum(tmax_day - tmin_day, 0.0)
    lam   = 2.501 - 0.002361 * Tmean          # latent heat of vaporization (MJ/kg), temp-dependent
    Ra_mm = compute_Ra(lat_rad_2d, doy) / lam  # MJ/m2/day -> mm/day equivalent
    return 0.0023 * Ra_mm * (Tmean + 17.8) * np.sqrt(TD)

print("ETo functions defined.")


lat_rad = np.radians(LAT_C)[:, None]                     # (cells, 1): the BIO code works on (lat, lon) = (cells, 1)
doys = dates.dayofyear.values
ETO = np.empty_like(TMAX)
for t in range(dates.size):
    ETO[t] = compute_ETo_day(TMAX[t][:, None], TMIN[t][:, None], lat_rad, doys[t])[:, 0]
print("ET0 done:", ETO.shape)

# global radiation estimated from temperature (FAO-56 eq. 50): Rs = kRs sqrt(Tmax - Tmin) Ra, kRs 0.16 inland / 0.19
# coastal (land cells with a sea cell among their 8 neighbours), capped at the clear-sky radiation 0.75 Ra (eq. 37)
if RAD_EST:
    sea = ~LAND
    near_sea = np.zeros_like(sea)
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            near_sea |= np.roll(np.roll(sea, di, axis=0), dj, axis=1)
    COAST = (LAND & near_sea)[I, J]
    KRS = np.where(COAST, 0.19, 0.16).astype(np.float32)
    SW_EST = np.empty_like(TMAX)
    for t in range(dates.size):
        ra = compute_Ra(lat_rad, doys[t])[:, 0]
        SW_EST[t] = np.minimum(KRS * np.sqrt(np.maximum(TMAX[t] - TMIN[t], 0.0)) * ra, 0.75 * ra)
    print(f"estimated radiation done ({int(COAST.sum())} coastal cells, kRs 0.19)")

## 3. BIO variables (unchanged definitions)

In [ ]:
def compute_bio_variables(TMAX, TMIN, PR, SW, ETO, dates,
                          sowing_doy, anthesis_doy, maturity_doy):
    """
    12 BIO variables averaged over harvest years 1982-2015.
    Calendar days are counted on a 2-year axis (year hy-1 then hy). A season that ends
    in the calendar year of harvest (maturity_doy <= 365) uses year hy; a season that
    crosses New Year (maturity_doy > 365) starts in year hy-1 and ends in year hy.
    Seasons whose start year is missing from the data are skipped for that harvest year
    (each pixel is averaged over the years where it could be computed).
    """
    nlat, nlon = sowing_doy.shape
    yr_all  = dates.year.values
    doy_all = dates.dayofyear.values
    mon_all = dates.month.values
    harvest_years = list(range(int(PERIOD[0]) + 1, int(PERIOD[1]) + 1))
    bio_sum = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    bio_cnt = {i: np.zeros((nlat, nlon)) for i in range(1, 13)}
    valid   = ~np.isnan(sowing_doy) & ~np.isnan(maturity_doy)
    wrap    = valid & (maturity_doy > 365)
    yrs_present = set(yr_all)

    def ylen(y): return 366 if (y % 4 == 0 and (y % 100 != 0 or y % 400 == 0)) else 365

    for hy in harvest_years:
        print(f'  {hy}...', end=' ')
        if not (yr_all == hy).any(): print('skip'); continue
        idx = np.where((yr_all == hy) | (yr_all == hy - 1))[0]
        yr3  = yr_all[idx][:, None, None]
        doy3 = doy_all[idx][:, None, None]
        mon3 = mon_all[idx][:, None, None]
        t3   = doy3 + np.where(yr3 == hy, ylen(hy - 1), 0)          # day index on the 2-year axis
        shift = np.where(wrap, 0, ylen(hy - 1))[None]                # non-wrapping seasons live in year hy
        sow3 = np.nan_to_num(sowing_doy)[None] + shift
        ant3 = np.nan_to_num(anthesis_doy)[None] + shift
        mat3 = np.nan_to_num(maturity_doy)[None] + shift
        vp3  = valid[None] & (((hy - 1) in yrs_present) | ~wrap[None])

        p1 = (t3 >= sow3) & (t3 < ant3)
        p2 = (t3 >= ant3) & (t3 <= mat3)
        gs = (p1 | p2) & vp3; p1 = p1 & vp3; p2 = p2 & vp3

        tm = TMAX[idx]; tn = TMIN[idx]; pr = PR[idx]; sw = SW[idx]; et = ETO[idx]
        def pm(a, m): return np.nanmean(np.where(m, a, np.nan), axis=0)
        def acc(k, v):
            f = np.isfinite(v); bio_sum[k] += np.where(f, v, 0); bio_cnt[k] += f

        acc(1, pm(tm, p1)); acc(2, pm(tn, p1)); acc(3, pm(sw, p1))
        acc(4, pm(et, p1)); acc(5, pm(pr, p1))
        acc(6, pm(tm, p2)); acc(7, pm(tn, p2)); acc(8, pm(sw, p2))
        acc(9, pm(et, p2)); acc(10, pm(pr, p2))

        tmx = np.full((nlat, nlon), np.nan); tmn = np.full((nlat, nlon), np.nan)
        for m in range(1, 13):
            mm = gs & (mon3 == m)
            if not np.any(mm): continue
            tmx = np.fmax(tmx, np.nanmean(np.where(mm, tm, np.nan), axis=0))
            tmn = np.fmin(tmn, np.nanmean(np.where(mm, tn, np.nan), axis=0))
        acc(11, tmx); acc(12, tmn)
        print('ok')

    bio = {}
    for i in range(1, 13):
        with np.errstate(invalid='ignore', divide='ignore'):
            bio[i] = np.where(bio_cnt[i] > 0, bio_sum[i] / bio_cnt[i], np.nan)
        bio[i][~valid] = np.nan
    return bio

print('compute_bio_variables defined.')

In [ ]:
def fill_calendar(doy, known):
    # land cells without the crop: inverse-distance-weighted mean of the 5 nearest cropped cells (great-circle
    # distance; floor 0.5 deg as before). Circular in DOY is not needed: 0.1 gives sowing in 1..365 and maturity
    # = sowing + season length, so neighbouring values are on the same scale.
    from scipy.spatial import cKDTree
    la, lo = np.radians(LAT_C), np.radians(LON_C)
    X = np.column_stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)])
    out = doy.copy()
    todo = ~known
    if known.sum() == 0 or todo.sum() == 0:
        return out
    k = min(5, int(known.sum()))
    d, ix = cKDTree(X[known]).query(X[todo], k=k)
    d, ix = d.reshape(-1, k), ix.reshape(-1, k)
    deg = np.degrees(2 * np.arcsin(np.clip(d / 2, 0, 1)))  # chord -> angle (deg)
    w = 1.0 / np.maximum(deg, 0.5) ** 2
    out[todo] = (w * doy[known][ix]).sum(1) / w.sum(1)
    return out


BIO_NAMES = {
    1: 'Avg Tmax sowing-anthesis [degC]', 2: 'Avg Tmin sowing-anthesis [degC]',
    3: 'Avg solar rad sowing-anthesis [MJ/m2/d]', 4: 'Avg ETo sowing-anthesis [mm/d]',
    5: 'Avg precip sowing-anthesis [mm/d]', 6: 'Avg Tmax anthesis-maturity [degC]',
    7: 'Avg Tmin anthesis-maturity [degC]', 8: 'Avg solar rad anthesis-maturity [MJ/m2/d]',
    9: 'Avg ETo anthesis-maturity [mm/d]', 10: 'Avg precip anthesis-maturity [mm/d]',
    11: 'Tmax warmest month sowing-maturity [degC]', 12: 'Tmin coldest month sowing-maturity [degC]'}
SEASON_NAMES = {1: 'Avg Tmax sowing-maturity [degC]', 2: 'Avg Tmin sowing-maturity [degC]',
                3: 'Avg solar rad sowing-maturity [MJ/m2/d]', 4: 'Avg ETo sowing-maturity [mm/d]',
                5: 'Avg precip sowing-maturity [mm/d]', 11: BIO_NAMES[11], 12: BIO_NAMES[12]}


def to_grid(x):
    g = np.full(LAND.shape, np.nan, np.float32)
    g[I, J] = x
    return g


def split_doy(variant, crop, sow, mat):
    # start of phase 2 (DOY) for the variant and crop; without split, phase 1 = the whole season (split at maturity)
    rule = VARIANTS[variant]
    if rule is None:
        return mat.copy(), "no split (whole season)"
    base = crop.split("_")[0].split("-")[0]
    if base not in rule and "default" not in rule:
        raise KeyError(f"{variant}: no phase rule for {base}")
    kind, d = rule.get(base, rule.get("default"))
    split = mat - d if kind == "before_maturity" else sow + d
    return np.clip(split, sow + 1, mat - 1), f"{kind} {d} days"


summary = []
for crop in CROPS:
    cm = cu.crop_mask(CFG, crop)
    pres = cm["presence"].values[I, J] == 1
    sow0, mat0 = cm["sowing_doy"].values[I, J].astype(float), cm["maturity_doy"].values[I, J].astype(float)
    known = pres & np.isfinite(sow0) & np.isfinite(mat0)       # MIRCA calendar: every cell with the crop
    label = known & (cu.cell_area_frac(CFG, cm["area_ha"].values)[I, J] > CELL_AREA_THR) if CELL_AREA_THR else known
    if (label & ~TRAIN).any():
        raise ValueError(f"{crop}: {int((label & ~TRAIN).sum())} presence cells outside the training domain")
    sow, mat = fill_calendar(sow0, known), fill_calendar(mat0, known)
    print(f"{crop}: {int(known.sum())} cropped cells with a calendar, {NC - int(known.sum())} filled; "
          f"presences {int(label.sum())} ({ROUTE})")
    for variant in VARIANTS_RUN:
        t1 = time.time()
        ant, rule = split_doy(variant, crop, sow, mat)
        keys = [1, 2, 3, 4, 5, 11, 12] if VARIANTS[variant] is None else list(range(1, 13))
        names = SEASON_NAMES if VARIANTS[variant] is None else BIO_NAMES
        bio = compute_bio_variables(TMAX[:, :, None], TMIN[:, :, None], PR[:, :, None], SW[:, :, None], ETO[:, :, None],
                                    dates, sow[:, None], ant[:, None], mat[:, None])
        B = {f"BIO{k:02d}": bio[k][:, 0] for k in keys}
        rad_keys = [k for k in (3, 8) if k in keys] if RAD_EST else []
        if RAD_EST:
            bio_e = compute_bio_variables(TMAX[:, :, None], TMIN[:, :, None], PR[:, :, None], SW_EST[:, :, None], ETO[:, :, None],
                                          dates, sow[:, None], ant[:, None], mat[:, None])  # only its radiation means are kept
            B.update({f"BIO{k:02d}e": bio_e[k][:, 0] for k in rad_keys})
        long = {f"BIO{k:02d}": names[k] for k in keys}
        long.update({f"BIO{k:02d}e": names[k].replace("solar rad", "solar rad estimated (FAO-56 eq. 50)") for k in rad_keys})

        ds = xr.Dataset({c: (("lat", "lon"), to_grid(x), {"long_name": long[c]}) for c, x in B.items()},
                        coords={"lat": GRID["lat"], "lon": GRID["lon"]})
        for n, x in (("sowing_doy", sow), ("phase2_start_doy", ant), ("maturity_doy", mat)):
            ds[n] = (("lat", "lon"), to_grid(x), {"long_name": f"{n} used (MIRCA2000 no threshold; gap-filled off the crop)"})
        ds["presence"] = (("lat", "lon"), to_grid(label.astype(np.float32)), {"long_name": f"MIRCA2000 presence (label 1), {ROUTE}"})
        ds["train_domain"] = (("lat", "lon"), to_grid(TRAIN.astype(np.float32)), {"long_name": f"SVM training domain ({TRAIN_DOMAIN}, {ROUTE})"})
        ds.attrs.update({"description": f"BIO variables, {CLIMATE} {PERIOD[0]}-{PERIOD[1]}, EC-Earth3 grid, land cells, crop={crop}",
                         "variant": variant, "phase_rule": rule, "route": ROUTE})
        ds.to_netcdf(OUT[variant] / f"BIOvariables_{TAG}_{crop}.nc")

        cols = list(B)
        df = pd.DataFrame({"lat": LAT_C, "lon": LON_C, **B, "label": label.astype(int)})[TRAIN]
        df = df[np.isfinite(df[cols]).all(axis=1)]
        df.to_csv(OUT[variant] / f"training_dataset_{TAG}_{crop}.csv", index=False)
        summary.append({"variant": variant, "crop": crop, "phase rule": rule, "training cells": len(df),
                        "label 1": int(df.label.sum()), "label 0": int((df.label == 0).sum()),
                        "minutes": round((time.time() - t1) / 60, 1)})
        print(f"  {variant:8s} {rule:28s} saved ({(time.time() - t1) / 60:.1f} min)")
summary = pd.DataFrame(summary).set_index(["variant", "crop"])
summary.to_csv(OUT0 / f"summary_{TAG}.csv")
display(summary)

## 4. Checks: maps of the 12 variables, and values at a few places

In [ ]:
CHECK, VCHECK = CROPS[0], VARIANTS_RUN[0]
ds = xr.open_dataset(OUT[VCHECK] / f"BIOvariables_{TAG}_{CHECK}.nc")
fig, axes = plt.subplots(4, 3, figsize=(18, 14))
for k, ax in enumerate(axes.ravel(), 1):
    ds[f"BIO{k:02d}"].plot(ax=ax, cmap="viridis", robust=True)
    ax.set_title(f"BIO{k:02d} — {BIO_NAMES[k]}", fontsize=9); ax.set_xlabel(""); ax.set_ylabel("")
plt.suptitle(f"BIO variables — {CLIMATE} {PERIOD[0]}-{PERIOD[1]}, EC-Earth3 grid ({CHECK}, {VCHECK})")
plt.tight_layout(); plt.savefig(OUT[VCHECK] / f"BIOvariables_maps_{TAG}_{CHECK}.png", dpi=90); plt.show()

# the same places in the three variants: phase 2 start and a few variables
places = {"France (47N, 2E)": (47, 2), "Ukraine (49N, 32E)": (49, 32), "Kansas (38N, 98W)": (38, -98),
          "Punjab (30N, 75E)": (30, 75), "Argentina (34S, 62W)": (-34, -62), "Sahara (25N, 10E)": (25, 10)}
rows = []
for crop in CROPS:
    for variant in VARIANTS_RUN:
        d = xr.open_dataset(OUT[variant] / f"BIOvariables_{TAG}_{crop}.nc")
        for name, (la, lo) in places.items():
            p = d.sel(lat=la, lon=lo, method="nearest")
            rows.append({"crop": crop, "variant": variant, "place": name, "presence": float(p.presence),
                         "sowing": float(p.sowing_doy), "phase 2 from": float(p.phase2_start_doy), "maturity": float(p.maturity_doy),
                         **{f"BIO{k:02d}": round(float(p[f'BIO{k:02d}']), 1) for k in (1, 5, 6, 10, 11) if f"BIO{k:02d}" in p}})
with pd.option_context("display.max_rows", 300, "display.width", 220):
    display(pd.DataFrame(rows).set_index(["crop", "place", "variant"]).sort_index().round(1))